# Decision Trees on vehicle attributes

**CSCI 5612 — Module 3, Project Part 3**

Same label as the Naive Bayes notebook — did this complaint involve a crash —
but predicted from the vehicle's characteristics instead of the narrative text.
Keeping the target identical makes the two directly comparable.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

pd.set_option("display.width", 200)
RANDOM_STATE, TEST_SIZE = 5612, 0.25

## 1. Impurity measures, written out

Both are zero for a pure group. Gini peaks at 0.5 for a two-class problem,
entropy at 1.0.

In [2]:
def gini(y):
    p = np.bincount(y, minlength=2) / len(y)
    return float(1 - (p ** 2).sum())

def entropy(y):
    p = np.bincount(y, minlength=2) / len(y)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def info_gain(y, mask, fn):
    n = len(y)
    weighted = len(y[mask])/n * fn(y[mask]) + len(y[~mask])/n * fn(y[~mask])
    return fn(y) - weighted, fn(y), weighted

print("a 50/50 group  -> gini", round(gini(np.array([0,1])), 4),
      "| entropy", round(entropy(np.array([0,1])), 4))
print("a pure group   -> gini", round(gini(np.array([0,0])), 4),
      "| entropy", round(entropy(np.array([0,0])), 4))

a 50/50 group  -> gini 0.5 | entropy 1.0
a pure group   -> gini 0.0 | entropy -0.0


## 2. Labelled feature matrix

Trees in scikit-learn need numeric input, so the categorical fields are one-hot encoded.

In [3]:
c = pd.read_csv("../data/clean/complaints_clean.csv.gz", low_memory=False)
my = pd.read_csv("../data/clean/vehicle_model_year.csv")
c = c.merge(my[["make","model","model_year","OverallRating","RolloverPossibility",
                "has_fcw","has_ldw","n_recalls"]],
            on=["make","model","model_year"], how="inner")
c["label"] = np.where(c["crash"], "Crash", "No crash")
top = c["component_primary"].value_counts().head(12).index
c["component"] = np.where(c["component_primary"].isin(top), c["component_primary"], "OTHER")

feat = pd.DataFrame({
    "vehicle_age": c["vehicle_age_at_incident"], "days_to_report": c["days_to_report"],
    "model_year": c["model_year"], "overall_rating": c["OverallRating"],
    "rollover_prob": c["RolloverPossibility"], "has_fcw": c["has_fcw"],
    "has_ldw": c["has_ldw"], "n_recalls": c["n_recalls"],
    "summary_words": c["summary"].fillna("").astype(str).str.split().str.len(),
})
feat = pd.concat([feat,
                  pd.get_dummies(c["segment"], prefix="seg").astype(int),
                  pd.get_dummies(c["component"], prefix="cmp").astype(int)], axis=1)
feat["label"] = c["label"].values
feat = feat.dropna().reset_index(drop=True)
print("feature matrix:", feat.shape)
feat.head(3)

feature matrix: (80615, 30)


,vehicle_age,days_to_report,model_year,overall_rating,rollover_prob,has_fcw,has_ldw,n_recalls,summary_words,seg_Electric,...,cmp_FUEL/PROPULSION SYSTEM,cmp_OTHER,cmp_POWER TRAIN,cmp_SERVICE BRAKES,cmp_STEERING,cmp_STRUCTURE,cmp_UNKNOWN OR OTHER,cmp_VEHICLE SPEED CONTROL,cmp_VISIBILITY/WIPER,label
0,10,1,2016,5.0,0.099,0,0,4,12,0,...,0,0,0,0,0,0,0,0,1,No crash
1,9,470,2016,5.0,0.099,0,0,4,74,0,...,0,0,0,0,0,0,0,0,0,No crash
2,10,3,2016,5.0,0.099,0,0,4,49,0,...,0,0,0,0,0,0,1,0,0,No crash


## 3. Disjoint, stratified split

In [4]:
X, y = feat.drop(columns=["label"]), feat["label"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE,
                                      random_state=RANDOM_STATE, stratify=y)
print(f"train {len(Xtr):,}   test {len(Xte):,}")
print("rows in both:", len(set(Xtr.index) & set(Xte.index)))
assert not (set(Xtr.index) & set(Xte.index))

train 60,461   test 20,154
rows in both: 0


## 4. A worked split, scored by hand

Information gain is the parent's impurity minus the size-weighted impurity of
the two children it would create.

In [5]:
yb = (ytr == "Crash").astype(int).values
rows = []
for name, mask in [("summary_words <= 40", Xtr["summary_words"].values <= 40),
                   ("vehicle_age <= 2",     Xtr["vehicle_age"].values <= 2),
                   ("has_fcw == 1",         Xtr["has_fcw"].values == 1),
                   ("rollover_prob <= 0.15", Xtr["rollover_prob"].values <= 0.15)]:
    g, gp, gc = info_gain(yb, mask, gini)
    e, ep, ec = info_gain(yb, mask, entropy)
    rows.append({"split": name, "entropy_parent": ep, "entropy_children": ec,
                 "information_gain": e, "gini_gain": g})
pd.DataFrame(rows).sort_values("information_gain", ascending=False).round(6)

,split,entropy_parent,entropy_children,information_gain,gini_gain
1,vehicle_age <= 2,0.274399,0.269382,0.005017,0.000644
2,has_fcw == 1,0.274399,0.273424,0.000975,0.000125
0,summary_words <= 40,0.274399,0.273627,0.000772,0.000089
3,rollover_prob <= 0.15,0.274399,0.273898,0.000500,0.000063


## 5. Three different trees

In [6]:
specs = [
    ("Tree 1: gini, depth 3",  dict(criterion="gini", max_depth=3)),
    ("Tree 2: entropy, depth 5", dict(criterion="entropy", max_depth=5)),
    ("Tree 3: gini, depth 10, min_leaf 50",
     dict(criterion="gini", max_depth=10, min_samples_leaf=50)),
]
trees, rows = [], []
for name, kw in specs:
    clf = DecisionTreeClassifier(class_weight="balanced",
                                 random_state=RANDOM_STATE, **kw).fit(Xtr, ytr)
    p = clf.predict(Xte)
    rows.append({"model": name, "accuracy": accuracy_score(yte, p),
                 "f1_crash": f1_score(yte, p, pos_label="Crash", zero_division=0),
                 "leaves": clf.get_n_leaves()})
    trees.append((name, clf, p))
rows.append({"model": "Majority-class baseline",
             "accuracy": accuracy_score(yte, np.full(len(yte), "No crash")),
             "f1_crash": 0.0, "leaves": 1})
pd.DataFrame(rows).round(4)

,model,accuracy,f1_crash,leaves
0,"Tree 1: gini, depth 3",0.8822,0.2841,8
1,"Tree 2: entropy, depth 5",0.8721,0.2836,32
2,"Tree 3: gini, depth 10, min_leaf 50",0.7343,0.1997,170
3,Majority-class baseline,0.9528,0.0000,1


In [7]:
# The first tree, small enough to read as text
print(export_text(trees[0][1], feature_names=list(Xtr.columns), max_depth=2))

|--- cmp_AIR BAGS <= 0.50
|   |--- cmp_VEHICLE SPEED CONTROL <= 0.50
|   |   |--- cmp_SERVICE BRAKES <= 0.50
|   |   |   |--- class: No crash
|   |   |--- cmp_SERVICE BRAKES >  0.50
|   |   |   |--- class: Crash
|   |--- cmp_VEHICLE SPEED CONTROL >  0.50
|   |   |--- n_recalls <= 3.50
|   |   |   |--- class: Crash
|   |   |--- n_recalls >  3.50
|   |   |   |--- class: Crash
|--- cmp_AIR BAGS >  0.50
|   |--- days_to_report <= 0.50
|   |   |--- has_fcw <= 0.50
|   |   |   |--- class: Crash
|   |   |--- has_fcw >  0.50
|   |   |   |--- class: Crash
|   |--- days_to_report >  0.50
|   |   |--- vehicle_age <= 2.50
|   |   |   |--- class: Crash
|   |   |--- vehicle_age >  2.50
|   |   |   |--- class: Crash



In [8]:
best = max(range(3), key=lambda i: rows[i]["f1_crash"])
print(classification_report(yte, trees[best][2], zero_division=0))
pd.DataFrame(confusion_matrix(yte, trees[best][2], labels=["No crash", "Crash"]),
             index=["actual No crash", "actual Crash"],
             columns=["pred No crash", "pred Crash"])

              precision    recall  f1-score   support

       Crash       0.20      0.50      0.28       951
    No crash       0.97      0.90      0.94     19203

    accuracy                           0.88     20154
   macro avg       0.59      0.70      0.61     20154
weighted avg       0.94      0.88      0.91     20154



,pred No crash,pred Crash
actual No crash,17309,1894
actual Crash,480,471


## 6. Why a deeper tree is not a better tree

There are infinitely many possible trees: every numeric feature offers a
threshold between each pair of adjacent values, features can be reused at any
depth, and a branch can be extended until every leaf holds one record. Training
is greedy precisely because searching that space exhaustively is intractable.

In [9]:
rows = []
for d in range(1, 21):
    clf = DecisionTreeClassifier(criterion="gini", max_depth=d,
                                 class_weight="balanced",
                                 random_state=RANDOM_STATE).fit(Xtr, ytr)
    rows.append({"depth": d,
                 "train_f1": f1_score(ytr, clf.predict(Xtr), pos_label="Crash", zero_division=0),
                 "test_f1":  f1_score(yte, clf.predict(Xte), pos_label="Crash", zero_division=0)})
pd.DataFrame(rows).round(4)

,depth,train_f1,test_f1
0,1,0.3404,0.3389
1,2,0.3302,0.3227
2,3,0.2929,0.2841
3,4,0.2978,0.2877
4,5,0.2822,0.2705
5,6,0.3067,0.2893
6,7,0.3134,0.2871
7,8,0.2039,0.1797
8,9,0.2214,0.1896
9,10,0.2344,0.1946


## 7. What this says about the topic

Predicting a crash from the owner's words works reasonably well (F1 ≈ 0.61).
Predicting the same crash from the vehicle's own characteristics barely works
(F1 ≈ 0.28). Same label, same complaints, same split discipline — the only
difference is what each model was allowed to look at.

So whether an incident ended in a collision is mostly a property of **what
happened**, not of **what was driving**. The crash-test rating — the headline
number a buyer is given — ranks near the bottom of what the tree found useful.
The one attribute that does carry signal is which part failed: the tree's first
question is whether the complaint concerns airbags.

Full discussion: <https://rajeshwar987.github.io/csci5612-project/dectrees.html>